# Exercise 1

Testing random numbers and seeing if they look ok or not.

## Imports and helper stuff

Just normal imports first. If `scipy` works I use it for p-values, if not then I still keep the main test numbers.

In [ ]:
import math
import os
import random
import statistics

try:
    import numpy as np
except ImportError:
    np = None

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from scipy import stats
except ImportError:
    stats = None

PLOT_DIR = "Exercise1/pics"


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_fig(fig, filename):
    if plt is None:
        print("matplotlib not available, skipping figure:", filename)
        return
    ensure_dir(PLOT_DIR)
    out_path = os.path.join(PLOT_DIR, filename)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print("saved", out_path)


ensure_dir(PLOT_DIR)

print("numpy available:", np is not None)
print("scipy available:", stats is not None)
print("matplotlib available:", plt is not None)
print("plot folder:", PLOT_DIR)


## LCG

Using

\[
x_i = (a x_{i-1} + c) \bmod M
\]

and then `U_i = x_i / M`. I kept both states and uniforms bc the states are handy when checking if it starts looping fast.

In [ ]:
def lcg(a, c, M, x0, n):
    x = x0
    states = []
    values = []
    for _ in range(n):
        x = (a * x + c) % M
        states.append(x)
        values.append(x / M)
    return states, values


def estimate_period(states):
    seen = {}
    for i, x in enumerate(states):
        if x in seen:
            return i - seen[x]
        seen[x] = i
    return None


test_states, test_values = lcg(a=5, c=1, M=16, x0=3, n=20)
print("first states:", test_states[:12])
print("first values:", [round(x, 4) for x in test_values[:12]])
print("estimated period from this short run:", estimate_period(test_states))


## Test functions

Bit boring but needed. I just put the checks here so I dont repeat the same stuff 3 times.

In [ ]:
def normal_tail_pvalue(z):
    if stats is not None:
        return 2 * (1 - stats.norm.cdf(abs(z)))
    return math.erfc(abs(z) / math.sqrt(2))


def count_bins(values, k=10):
    counts = [0] * k
    for u in values:
        idx = min(int(u * k), k - 1)
        counts[idx] += 1
    return counts


def histogram_plot(values, title, filename, bins=20):
    counts = count_bins(values, bins)
    if plt is not None:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(values, bins=bins, range=(0, 1), color="steelblue", edgecolor="black", alpha=0.85)
        ax.set_title(title)
        ax.set_xlabel("u")
        ax.set_ylabel("count")
        ax.grid(alpha=0.2)
        save_fig(fig, filename)
        plt.show()
        plt.close(fig)
    return counts


def chi_square_test(values, k=10):
    n = len(values)
    counts = count_bins(values, k)
    expected = n / k
    statistic = sum((obs - expected) ** 2 / expected for obs in counts)
    df = k - 1
    if stats is not None:
        pvalue = float(stats.chi2.sf(statistic, df))
    else:
        pvalue = None
    return {
        "statistic": statistic,
        "df": df,
        "pvalue": pvalue,
        "reject_5pct": (pvalue < 0.05) if pvalue is not None else (statistic > 16.919),
        "counts": counts,
    }


def ks_test_uniform(values):
    n = len(values)
    sorted_vals = sorted(values)
    d_plus = max((i + 1) / n - u for i, u in enumerate(sorted_vals))
    d_minus = max(u - i / n for i, u in enumerate(sorted_vals))
    d_stat = max(d_plus, d_minus)
    critical_5pct = 1.36 / math.sqrt(n)
    if stats is not None:
        _, pvalue = stats.kstest(sorted_vals, "uniform")
        pvalue = float(pvalue)
    else:
        pvalue = None
    return {
        "D+": d_plus,
        "D-": d_minus,
        "D": d_stat,
        "critical_5pct": critical_5pct,
        "pvalue": pvalue,
        "reject_5pct": (pvalue < 0.05) if pvalue is not None else (d_stat > critical_5pct),
    }


def scatter_successive(values, title, filename, max_points=2500):
    if plt is None:
        return
    x = values[:-1]
    y = values[1:]
    if len(x) > max_points:
        x = x[:max_points]
        y = y[:max_points]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(x, y, s=8, alpha=0.5, color="darkred")
    ax.set_title(title)
    ax.set_xlabel("$U_i$")
    ax.set_ylabel("$U_{i+1}$")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.15)
    save_fig(fig, filename)
    plt.show()
    plt.close(fig)


def runs_test_above_below(values, cutoff=0.5):
    signs = []
    for u in values:
        if u > cutoff:
            signs.append(1)
        elif u < cutoff:
            signs.append(0)
    n1 = sum(signs)
    n2 = len(signs) - n1
    runs = 1
    for i in range(1, len(signs)):
        if signs[i] != signs[i - 1]:
            runs += 1
    mean_runs = 2 * n1 * n2 / (n1 + n2) + 1
    var_runs = (
        2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)
        / (((n1 + n2) ** 2) * (n1 + n2 - 1))
    )
    z = (runs - mean_runs) / math.sqrt(var_runs)
    pvalue = normal_tail_pvalue(z)
    return {
        "runs": runs,
        "n1": n1,
        "n2": n2,
        "mean": mean_runs,
        "variance": var_runs,
        "z": z,
        "pvalue": pvalue,
        "reject_5pct": pvalue < 0.05,
    }


def up_down_runs_test(values):
    directions = []
    for i in range(len(values) - 1):
        if values[i + 1] > values[i]:
            directions.append(1)
        elif values[i + 1] < values[i]:
            directions.append(0)
    runs = 1
    for i in range(1, len(directions)):
        if directions[i] != directions[i - 1]:
            runs += 1
    n = len(directions)
    mean_runs = (2 * n - 1) / 3
    variance = (16 * n - 29) / 90
    z = (runs - mean_runs) / math.sqrt(variance)
    pvalue = normal_tail_pvalue(z)
    return {
        "runs": runs,
        "n_directions": n,
        "z": z,
        "pvalue": pvalue,
        "reject_5pct": pvalue < 0.05,
    }


def lag_correlation_tests(values, lags=(1, 2, 3, 5, 10, 20)):
    n = len(values)
    out = []
    for h in lags:
        products = [values[i] * values[i + h] for i in range(n - h)]
        c_h = sum(products) / (n - h)
        variance = 7 / (144 * n)
        z = (c_h - 0.25) / math.sqrt(variance)
        pvalue = normal_tail_pvalue(z)
        out.append(
            {
                "lag": h,
                "C_h": c_h,
                "z": z,
                "pvalue": pvalue,
                "reject_5pct": pvalue < 0.05,
            }
        )
    return out


def show_result_dict(name, results):
    print(name)
    for key, value in results.items():
        print(f"  {key}: {value}")


## Small runner

This just runs the same set of checks for each generator so the notebook stays kinda managable.

In [ ]:
def analyze_values(values, label, hist_name, scatter_name, visual_comment):
    print(label)
    print("-" * len(label))
    print("n =", len(values))
    print("mean =", round(statistics.mean(values), 6))
    print("variance =", round(statistics.pvariance(values), 6))
    print("min/max =", round(min(values), 6), round(max(values), 6))

    counts = histogram_plot(values, f"{label} histogram", hist_name, bins=20)
    scatter_successive(values, f"{label} successive pairs", scatter_name)

    chi = chi_square_test(values, k=20)
    ks = ks_test_uniform(values)
    runs = runs_test_above_below(values)
    updown = up_down_runs_test(values)
    corr = lag_correlation_tests(values)

    print("\nchi-square:")
    print(chi)
    print("\nKS test:")
    print(ks)
    print("\nruns test:")
    print(runs)
    print("\nup/down runs test:")
    print(updown)
    print("\nlag correlations:")
    for row in corr:
        print(row)

    summary = {
        "generator": label,
        "chi_square_pvalue": chi["pvalue"],
        "ks_pvalue": ks["pvalue"],
        "runs_pvalue": runs["pvalue"],
        "updown_pvalue": updown["pvalue"],
        "worst_corr_pvalue": min(row["pvalue"] for row in corr),
        "visual_comment": visual_comment,
    }
    return {
        "counts": counts,
        "chi_square": chi,
        "ks": ks,
        "runs": runs,
        "updown": updown,
        "corr": corr,
        "summary": summary,
    }


## Bad LCG

Starting with an obviously bad one first.

- `a = 5`
- `c = 1`
- `M = 16`
- `x0 = 3`

Tiny modulus so it should go wrong pretty fast.

In [ ]:
n = 10_000
bad_params = {"a": 5, "c": 1, "M": 16, "x0": 3}
bad_states, bad_values = lcg(n=n, **bad_params)

print("bad generator first 20 states:", bad_states[:20])
print("unique states among first 10,000 values:", len(set(bad_states)))
print("estimated period:", estimate_period(bad_states))

bad_results = analyze_values(
    bad_values,
    label="Bad LCG",
    hist_name="lcg_bad_hist.png",
    scatter_name="lcg_bad_scatter.png",
    visual_comment="very strong structure, obvious repetition",
)


This one is bad pretty much imidiately. The period is tiny and the scatter plot gives it away fast.

## Another bad-ish one

Just tried one more small setup so it wasnt only one weird example.

In [ ]:
bad2_params = {"a": 6, "c": 0, "M": 31, "x0": 7}
bad2_states, bad2_values = lcg(n=120, **bad2_params)
print("first 20 states:", bad2_states[:20])
print("number of unique states in first 120 draws:", len(set(bad2_states)))
print("estimated period:", estimate_period(bad2_states))


## Better LCG

Then I tried a more standard choice.

- `a = 1103515245`
- `c = 12345`
- `M = 2**31`
- `x0 = 123456789`

Not saying its perfect or anything, just way less awful.

In [ ]:
good_params = {"a": 1103515245, "c": 12345, "M": 2**31, "x0": 123456789}
good_states, good_values = lcg(n=n, **good_params)

print("good generator first 5 states:", good_states[:5])
print("unique states among first 10,000 values:", len(set(good_states)))
print("estimated period from this sample:", estimate_period(good_states))

good_results = analyze_values(
    good_values,
    label="Better LCG",
    hist_name="lcg_good_hist.png",
    scatter_name="lcg_good_scatter.png",
    visual_comment="looks fairly uniform, no obvious pattern at this scale",
)


For one sample I wouldnt panic if a decent generator fails one thing by chance. What matters more is whether there is a clear pattern or if it keeps failing over and over.

## Python generator

For comparison I also used Python `random.Random` with a fixed seed.

In [ ]:
system_rng = random.Random(20240615)
system_values = [system_rng.random() for _ in range(n)]

system_results = analyze_values(
    system_values,
    label="Python random()",
    hist_name="system_hist.png",
    scatter_name="system_scatter.png",
    visual_comment="looks good visually, no clear lattice pattern here",
)


## Correlation plot

Just plotting the lag z-scores here. Big absolute values would be sus.

In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for label, result, color in [
        ("Bad LCG", bad_results, "firebrick"),
        ("Better LCG", good_results, "steelblue"),
        ("Python random()", system_results, "darkgreen"),
    ]:
        lags = [row["lag"] for row in result["corr"]]
        zscores = [row["z"] for row in result["corr"]]
        ax.plot(lags, zscores, marker="o", label=label, color=color)
    ax.axhline(1.96, color="gray", linestyle="--", linewidth=1)
    ax.axhline(-1.96, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("lag")
    ax.set_ylabel("z-score")
    ax.set_title("Lag correlation test")
    ax.legend()
    ax.grid(alpha=0.2)
    save_fig(fig, "lag_correlation_plot.png")
    plt.show()
    plt.close(fig)


## Final table

Putting the p-values and quick comments together so its easier to compare.

In [ ]:
final_table = []

for result in [bad_results, good_results, system_results]:
    row = result["summary"].copy()
    conclusion = "bad"
    if "Better" in row["generator"]:
        conclusion = "looks okay in this experiment"
    if "Python" in row["generator"]:
        conclusion = "best of the three here"
    row["conclusion"] = conclusion
    final_table.append(row)

for row in final_table:
    print(row)


## Repeated testing

One sample is not enough really, so I repeated the main tests a bunch of times and counted how often they reject.

In [ ]:
def sample_bad_lcg(rep, n=10_000):
    x0 = (3 + rep) % 16
    if x0 == 0:
        x0 = 1
    _, values = lcg(a=5, c=1, M=16, x0=x0, n=n)
    return values


def sample_good_lcg(rep, n=10_000):
    x0 = 123456789 + 7919 * rep
    _, values = lcg(a=1103515245, c=12345, M=2**31, x0=x0, n=n)
    return values


def sample_system(rep, n=10_000):
    rng = random.Random(10_000 + rep)
    return [rng.random() for _ in range(n)]


def repeated_pvalues(sample_fn, reps=60):
    chi_p = []
    ks_p = []
    for rep in range(reps):
        values = sample_fn(rep)
        chi_p.append(chi_square_test(values, k=20)["pvalue"])
        ks_p.append(ks_test_uniform(values)["pvalue"])
    return {"chi": chi_p, "ks": ks_p}


reps = 60
bad_repeat = repeated_pvalues(sample_bad_lcg, reps=reps)
good_repeat = repeated_pvalues(sample_good_lcg, reps=reps)
system_repeat = repeated_pvalues(sample_system, reps=reps)


def rejection_rate(pvalues, alpha=0.05):
    return sum(p < alpha for p in pvalues) / len(pvalues)


repeat_summary = {
    "Bad LCG": {
        "chi_reject_rate": rejection_rate(bad_repeat["chi"]),
        "ks_reject_rate": rejection_rate(bad_repeat["ks"]),
    },
    "Better LCG": {
        "chi_reject_rate": rejection_rate(good_repeat["chi"]),
        "ks_reject_rate": rejection_rate(good_repeat["ks"]),
    },
    "Python random()": {
        "chi_reject_rate": rejection_rate(system_repeat["chi"]),
        "ks_reject_rate": rejection_rate(system_repeat["ks"]),
    },
}

for key, value in repeat_summary.items():
    print(key, value)


In [ ]:
if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

    plot_data = [
        ("Bad LCG", bad_repeat, "firebrick"),
        ("Better LCG", good_repeat, "steelblue"),
        ("Python random()", system_repeat, "darkgreen"),
    ]

    for label, data, color in plot_data:
        axes[0].plot(data["chi"], marker="o", linestyle="", alpha=0.6, label=label, color=color)
        axes[1].plot(data["ks"], marker="o", linestyle="", alpha=0.6, label=label, color=color)

    for ax, title in zip(axes, ["Chi-square p-values", "KS p-values"]):
        ax.axhline(0.05, color="gray", linestyle="--", linewidth=1)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel("replication")
        ax.set_title(title)
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("p-value")
    axes[1].legend()
    fig.suptitle("Repeated testing across samples")
    save_fig(fig, "pvalue_repeated_tests.png")
    plt.show()
    plt.close(fig)


For a good generator you still expect some random rejections at the 5% level, so zero rejections forever would also be a bit weird. The bad one should fail a lot more and also look bad in the plots.

## End

The tiny-modulus LCG is clearly bad. It repeats too fast and the structure shows up right away.

The better LCG looks alright in this small check, and Python's generator also looks fine. Main point is probly that one sample and one test is not enough, so doing a few diff checks is safer.